In [2]:
import sys
import math
sys.path.append('../../python/')  
from periphery import logicGate
from periphery import constant
print(constant.INV)

0


In [6]:
class Technology:
    def __init__(self, node_nm=45, roadmap='HP', transistor_type='conventional'):
        self.node_nm = node_nm
        self.roadmap = roadmap
        self.transistor_type = transistor_type
        self.initialized = False
        self.params = {}
        self._initialize()

    def _initialize(self):
        if self.initialized:
            print("Warning: Already initialized!")
            return
        if self.transistor_type == 'conventional':
            if self.node_nm == 45 and self.roadmap == 'HP':
                vdd = 1.0
                vth = 0.18
                phyGateLength = 45e-9
                capIdealGate = 4e-10
                capFringe = 5e-10
                capOverlap = capIdealGate * 0.2 if self.node_nm >= 22 else 0.0

                # Junction cap model (from BSIM4)
                buildInPotential = 0.9
                cjd = 1e-3
                cjswd = 2.5e-10
                cjswgd = 0.5e-10
                mjd = 0.5
                mjswd = 0.33
                mjswgd = 0.33
                #/* Properties not used so far */
                capPolywire = 0.0;	#/* TO-DO: we need to find the values */

                capJunction = cjd / pow(1 + vdd / buildInPotential, mjd)
                capSidewall = cjswd / pow(1 + vdd / buildInPotential, mjswd)
                capDrainToChannel = cjswgd / pow(1 + vdd / buildInPotential, mjswgd)

                # On-current (A/m)
                currentOnNmos = [
                    1.27e3, 1.24e3, 1.22e3, 1.19e3, 1.16e3, 
                    1.13e3, 1.11e3, 1.08e3, 1.05e3, 1.02e3, 1.00e3
                ]

                currentOnPmos = [
                    1.08e3, 1.04e3, 1.00e3, 0.96e3, 0.92e3, 
                    0.88e3, 0.85e3, 0.81e3, 0.78e3, 0.75e3, 0.72e3
                ]

                # Off-current (A/m)
                currentOffNmos = [
                    100.00e-3, 120.70e-3, 144.10e-3, 170.50e-3, 199.80e-3, 
                    232.30e-3, 268.00e-3, 307.10e-3, 349.50e-3, 395.40e-3, 444.80e-3
                ]

                currentOffPmos = [
                    100.20e-3, 118.70e-3, 139.30e-3, 162.00e-3, 186.80e-3, 
                    213.90e-3, 243.30e-3, 274.90e-3, 308.90e-3, 345.20e-3, 383.80e-3
                ]

                # Interpolation to full 0-100
                currentOnNmos = self._interpolate_full(currentOnNmos)
                currentOnPmos = self._interpolate_full(currentOnPmos)
                currentOffNmos = self._interpolate_full(currentOffNmos)
                currentOffPmos = self._interpolate_full(currentOffPmos)

                self.params = {
                    'vdd': vdd,
                    'vth': vth,
                    'phyGateLength': phyGateLength,
                    'capIdealGate': capIdealGate,
                    'capFringe': capFringe,
                    'capOverlap': capOverlap,
                    'capJunction': capJunction,
                    'capSidewall': capSidewall,
                    'capDrainToChannel': capDrainToChannel,
                    'effectiveResistanceMultiplier': 1.0,  # could be adjusted based on design
                    'current_gmNmos': 1e-3,  # conductance for gm unit is μS/μm
                    'current_gmPmos': 0.8e-3,
                    'currentOnNmos': currentOnNmos,
                    'currentOnPmos': currentOnPmos,
                    'currentOffNmos': currentOffNmos,
                    'currentOffPmos': currentOffPmos,
                    'pnSizeRatio': 2.0,
                    'transistorType': self.transistor_type,   
                    'capPolywire': capPolywire,  
                    'featureSize': self.node_nm * 1e-9       
                }

            elif self.node_nm == 14 and self.roadmap == 'LSTP':
                vdd = 0.8
                vth = 0.45
                phyGateLength = 14e-9
                capIdealGate = 2.5e-10
                capFringe = 3e-10
                capOverlap = 0.0  # don't use overlap cap in finFET nodes

                buildInPotential = 0.9
                cjd = 1e-3
                cjswd = 2.5e-10
                cjswgd = 0.5e-10
                mjd = 0.5
                mjswd = 0.33
                mjswgd = 0.33
                #/* Properties not used so far */
                capPolywire = 0.0;	#/* TO-DO: we need to find the values */

                capJunction = cjd / pow(1 + vdd / buildInPotential, mjd)
                capSidewall = cjswd / pow(1 + vdd / buildInPotential, mjswd)
                capDrainToChannel = cjswgd / pow(1 + vdd / buildInPotential, mjswgd)

                currentOnNmos = [600e-6 - i * 2e-6 for i in range(0, 101, 10)]
                currentOnPmos = [500e-6 - i * 1.5e-6 for i in range(0, 101, 10)]
                currentOffNmos = [10e-9 + i * 0.1e-9 for i in range(0, 101, 10)]
                currentOffPmos = [8e-9 + i * 0.1e-9 for i in range(0, 101, 10)]

                currentOnNmos = self._interpolate_full(currentOnNmos)
                currentOnPmos = self._interpolate_full(currentOnPmos)
                currentOffNmos = self._interpolate_full(currentOffNmos)
                currentOffPmos = self._interpolate_full(currentOffPmos)

                self.params = {
                    'vdd': vdd,
                    'vth': vth,
                    'phyGateLength': phyGateLength,
                    'capIdealGate': capIdealGate,
                    'capFringe': capFringe,
                    'capOverlap': capOverlap,
                    'capJunction': capJunction,
                    'capSidewall': capSidewall,
                    'capDrainToChannel': capDrainToChannel,
                    'effectiveResistanceMultiplier': 1.2,
                    'current_gmNmos': 2.0e-3,
                    'current_gmPmos': 1.6e-3,
                    'currentOnNmos': currentOnNmos,
                    'currentOnPmos': currentOnPmos,
                    'currentOffNmos': currentOffNmos,
                    'currentOffPmos': currentOffPmos,
                    'capPolywire': capPolywire,  
                    'pnSizeRatio': 2.0
                }
            else:
                raise ValueError(f"Unsupported node {self.node_nm}nm or roadmap {self.roadmap}")

            self.initialized = True

    def _interpolate_full(self, base):
        'gemerate a full 0-100 array from a base array with 10 steps'
        full = [0.0] * 101
        for i in range(0, 101, 10):
            full[i] = base[i // 10]
        for i in range(1, 100):
            if i % 10 != 0:
                low = (i // 10) * 10
                high = low + 10
                alpha = (i - low) / 10
                full[i] = full[low] * (1 - alpha) + full[high] * alpha
        return full

    def get_param(self, name):
        return self.params.get(name, None)

    def interpolate_current(self, name, bias_index):
        'Interpolate current values based on bias index (0-100)'
        if name not in self.params:
            return None
        arr = self.params[name]
        i_low = int(bias_index)
        i_high = min(100, i_low + 1)
        alpha = bias_index - i_low
        return arr[i_low] * (1 - alpha) + arr[i_high] * alpha

    def print_summary(self):
        print(f"Tech Node: {self.node_nm}nm, Roadmap: {self.roadmap}")
        for k, v in self.params.items():
            if isinstance(v, list):
                print(f"{k}: {v[:3]} ... {v[-3:]}")
            else:
                print(f"{k}: {v:.3e}" if isinstance(v, float) else f"{k}: {v}")


In [3]:
config = {
    'mode': 'memory',
    'technology': '65',
    'device_type': 'SRAM',
    'frequency': 1e9,            # 1 GHz
    'precision_mu': 6,        # Device variation mean
    'precision_sigma': 2,    # Device variation stddev
    'dataset': 'CIFAR-10',
    'temperature': 350,  # Kelvin
    'model': 'ResNet18',
    'distribution_file': "../../../DATA/customized_gaussian_current.csv"
}

In [4]:


MIN_NMOS_SIZE = 1.0
# MAX_TRANSISTOR_HEIGHT = 2.5


class SRAMWriteDriver:
    def __init__(self, config, tech, num_col, activity_col_write, num_write_cell_per_op):
        self.config = config
        self.tech = tech
        self.initialized = False
        self.initialize(num_col, activity_col_write, num_write_cell_per_op)

    def initialize(self, num_col, activity_col_write, num_write_cell_per_op):
        self.num_col = num_col
        self.activity_col_write = activity_col_write
        self.num_write_cell_per_op = num_write_cell_per_op
        self.width_inv_n = constant.MIN_NMOS_SIZE * self.tech.get_param('featureSize')
        self.width_inv_p = self.tech.get_param('pnSizeRatio') * self.width_inv_n
        self.max_transistor_height = constant.MAX_TRANSISTOR_HEIGHT * self.tech.get_param('featureSize')
        self.initialized = True

    def calculate_area(self, new_height=0, new_width=0, option="NONE"):
        w_inv, h_inv, _ = logicGate.calculate_logicgate_area(gateType = constant.INV, num_Input = 1, width_NMOS = self.width_inv_n, width_PMOS = self.width_inv_p, height_Transistor_Region = self.max_transistor_height, tech = self.tech)
        w_nmos, h_nmos, _ = logicGate.calculate_logicgate_area(gateType = constant.INV, num_Input = 1, width_NMOS = self.width_inv_n, width_PMOS = 0, height_Transistor_Region = self.max_transistor_height, tech = self.tech)
        h_unit = h_inv + h_nmos
        w_unit = max(w_inv, w_nmos) * 2
        if new_width != 0 and option == "NONE":
            num_unit_per_row = int(new_width / w_unit)
            num_unit_per_row = min(num_unit_per_row, self.num_col)
            num_row_unit = math.ceil(self.num_col / num_unit_per_row) #in case we don't have enough width to fit all columns
            self.width = new_width
            self.height = num_row_unit * h_unit
        else:
            self.width = self.num_col * w_unit
            self.height = h_unit
        self.area = self.width * self.height
        self.cap_inv_input, self.cap_inv_output = logicGate.calculate_logicgate_cap(constant.INV, 1, self.width_inv_n, self.width_inv_p, h_inv, self.tech)
        self.cap_nmos_drain = logicGate.calculate_drain_cap('nmos', self.width_inv_n, self.max_transistor_height, self.tech)
        return {"height": self.height, "width": self.width, "area": self.area}

    def calculate_latency(self, ramp_input, cap_load, res_load, num_write):
        #first stage pull-up inv
        res_pull_up = logicGate.calculate_on_resistance(self.width_inv_p, constant.PMOS, self.config['temperature'], self.tech)
        tr = res_pull_up * (self.cap_inv_output + self.cap_inv_input + self.cap_nmos_drain)
        gm = logicGate.calculate_transconductance(self.width_inv_p, constant.PMOS, self.tech)
        beta = 1 / (res_pull_up * gm)
        delay1, rampInvOutput1 = logicGate.horowitz(tr, beta, ramp_input)

        #second stage pull-down inv
        res_pull_down = logicGate.calculate_on_resistance(self.width_inv_n, constant.NMOS, self.config['temperature'], self.tech)
        tr2 = res_pull_down * (self.cap_nmos_drain + self.cap_inv_output)
        gm2 = logicGate.calculate_transconductance(self.width_inv_n, constant.NMOS, self.tech)
        beta2 = 1 / (res_pull_down * gm2)
        delay2, rampInvOutput2 = logicGate.horowitz(tr2, beta2, rampInvOutput1)

        #third stage nmos pass transistor
        res_nmos = logicGate.calculate_on_resistance(self.width_inv_n, constant.NMOS, self.config['temperature'], self.tech)
        tr3 = res_nmos * (cap_load + self.cap_nmos_drain) + res_load * cap_load / 2
        gm3 = logicGate.calculate_transconductance(self.width_inv_n, constant.NMOS, self.tech)
        beta3 = 1 / (res_nmos * gm3)
        delay3, _ = logicGate.horowitz(tr3, beta3, rampInvOutput2)

        self.write_latency = (delay1 + delay2 + delay3) * num_write
        return self.write_latency

    def calculate_power(self, num_write):
        leakage = logicGate.calculate_logicgate_leakage(constant.INV, 1, self.width_inv_n, self.width_inv_p, self.config['temperature'], self.tech)
        self.leakage_power = leakage * self.tech.get_param('vdd') * 2 * self.num_col
        num_active = min(self.num_write_cell_per_op, self.num_col * self.activity_col_write)
        self.write_dynamic_energy = (self.cap_inv_input + self.cap_inv_output + self.cap_nmos_drain) * self.tech.get_param('vdd')**2 * num_active * num_write

    def print_property(self):
        print("SRAM Write Driver Properties:")
        print(f"  Area: {self.area:.3e} m^2")
        print(f"  Latency: {self.write_latency:.3e} s")
        print(f"  Leakage Power: {self.leakage_power:.3e} W")
        print(f"  Write Energy: {self.write_dynamic_energy:.3e} J")


In [7]:
tech45 = Technology(node_nm=45, roadmap='HP')
writedriver = SRAMWriteDriver(config=config, tech=tech45, num_col=64, activity_col_write=0.5, num_write_cell_per_op=64)

In [8]:
area_result = writedriver.calculate_area(new_height=0, new_width=0, option="NONE")
print("Area Result:", area_result)

Area Result: {'height': 2.5200000000000004e-06, 'width': 4.3776e-05, 'area': 1.1031552000000002e-10}


In [9]:
latency_result = writedriver.calculate_latency(ramp_input=1e20, cap_load=5e-15, res_load=1000, num_write=1)
print("Latency Result:", latency_result)

Latency Result: 4.5374761890265325e-11


In [10]:
power_result = writedriver.calculate_power(num_write=1)
print("Power Result:", power_result)
print(f"  Leakage: {writedriver.leakage_power:.3e} W")
print(f"  Write Dynamic Energy: {writedriver.write_dynamic_energy:.3e} J")

Power Result: None
  Leakage: 3.802e-06 W
  Write Dynamic Energy: 1.914e-14 J
